[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch14-ml-ops/03-phat-hien-troi.ipynb)

# Phát hiện trôi khi nhãn chưa về: PSI, KS và cái giá của nhãn

Chương 14 mở đầu bằng một câu hỏi: vì sao một hệ thống học máy có thể sẵn sàng hoàn hảo mà vẫn sai
hoàn toàn? Vì mọi kiểm tra sức khoẻ quen thuộc (độ trễ, thông lượng, thời gian hoạt động) vẫn xanh
trong khi độ chính xác giảm, và độ chính xác chỉ đo được khi nhãn chuẩn (ground truth) về. Chương đưa
ra hai cách nhìn sớm hơn: so phân phối của dữ liệu đang phục vụ với phân phối lúc huấn luyện, bằng chỉ
số ổn định phân phối (population stability index, PSI) hay kiểm định Kolmogorov-Smirnov
(Kolmogorov-Smirnov test, KS); và đếm xem cần bao nhiêu nhãn thì mới khẳng định được một mức giảm.

Sổ tay này chạy cả hai trên một mô hình thật, trên CPU, với bộ chữ số `load_digits`.

**Bạn sẽ làm:**
1. dựng lại "vấn đề tuần hai": đo độ trễ và độ chính xác của một mô hình trong tám tuần dữ liệu dịch
   chuyển dần;
2. tính PSI của bảng 20 bằng tay, nối nó với phân kỳ KL, và xem PSI lớn lên ra sao khi dịch chuyển
   lớn dần;
3. cho bốn kiểu dịch chuyển đi qua bốn bộ theo dõi không cần nhãn, xem bộ nào báo, bộ nào không thấy gì;
4. đo xem ngưỡng của bảng 19 báo nhầm bao thường xuyên khi không có trôi nào, theo cỡ cửa sổ;
5. kiểm napkin math 1.4 (1,000 nhãn) bằng một phép tính lực kiểm định, rồi thêm nhiễu nhãn và sự phụ
   thuộc giữa các nhãn;
6. tính lại bảng 23, rồi giấu một lát cắt 5 phần trăm hỏng nặng vào lưu lượng thật.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
from scipy import ndimage, optimize, stats
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định
XAM = "0.45"

# Ngưỡng của bảng 19: điểm khởi đầu, phải hiệu chuẩn.
PSI_CANH_BAO = sach(0.1, nguon="0.1 (cảnh báo)")
PSI_BAO_DONG = sach(0.25, nguon="PSI 0.25")
KS_NGUONG = sach(0.1, nguon="KS 0.1")

X, y = load_digits(return_X_y=True)
X = X / 16.0                                  # điểm ảnh về [0, 1]
thu_tu = np.random.default_rng(SEED).permutation(len(X))
hl, kt = thu_tu[:1078], thu_tu[1078:]         # 60 phần trăm để huấn luyện, phần còn lại là "lưu lượng"
mo_hinh = LogisticRegression(max_iter=2000).fit(X[hl], y[hl])
print(f"{len(hl)} ảnh huấn luyện, {len(kt)} ảnh làm lưu lượng phục vụ, 10 lớp, 64 điểm ảnh mỗi ảnh")


def xoay(anh, goc):
    """Xoay từng ảnh 8 × 8 một góc (độ). Đây là kiểu dịch chuyển của một camera bị lệch."""
    if goc == 0:
        return anh.copy()
    return np.clip(np.stack([ndimage.rotate(a.reshape(8, 8), goc, reshape=False, order=1, mode="grid-constant").ravel()
                             for a in anh]), 0, 1)

## 1. Vấn đề tuần hai

Kịch bản minh hoạ của chương là một hệ thống dự báo nhu cầu cho dịch vụ gọi xe. Tuần một, độ chính
xác đạt 94 phần trăm và độ trễ phân vị 99 là 15 ms. Tới tuần bốn, độ chính xác còn 88 phần trăm,
nhưng chỉ số hạ tầng không cho thấy gì bất thường. Tới tuần tám, một quản lý sản phẩm mới nhận ra việc
điều phối tài xế kém hiệu quả: mô hình chưa thích nghi với một đợt khuyến mãi của đối thủ. Lẽ ra phải
huấn luyện lại từ sáu tuần trước.

Ô dưới dựng lại kịch bản trên mô hình hồi quy logistic vừa huấn luyện. Mỗi tuần là 400 yêu cầu lấy từ
phần lưu lượng; từ tuần ba, ảnh bị xoay thêm 3 độ mỗi tuần (giả định của sổ tay). Mỗi tuần đo độ trễ
của 300 yêu cầu đơn lẻ và độ chính xác của cả 400.

**Dự đoán:** độ trễ phân vị 99 tuần tám so với tuần một thế nào?

In [ ]:
print("kịch bản minh hoạ của sách:")
print(f"  tuần một: độ chính xác {sach(94, nguon='94 phần trăm → 88 phần trăm · 15 ms phân vị 99')} phần trăm,"
      f" độ trễ phân vị 99 {sach(15, nguon='94 phần trăm → 88 phần trăm · 15 ms phân vị 99')} ms")
print(f"  tuần bốn: độ chính xác {sach(88, nguon='94 phần trăm → 88 phần trăm · 15 ms phân vị 99')} phần trăm,"
      " hạ tầng không báo gì")

for _ in range(50):                            # làm nóng trước khi đo
    mo_hinh.predict_proba(X[kt[:1]])

cac_tuan = np.arange(1, 9)
goc_tuan = 3.0 * np.maximum(0, cac_tuan - 2)   # giả định: lệch dần từ tuần ba
do_cx_tuan, p99_tuan = [], []
for so, goc in zip(cac_tuan, goc_tuan):
    r = np.random.default_rng(SEED + int(so))
    chon = r.choice(kt, 400)
    Xb = xoay(X[chon], goc)
    do_cx_tuan.append(100 * float((mo_hinh.predict(Xb) == y[chon]).mean()))
    tre = []
    for i in range(300):
        t0 = time.perf_counter()
        mo_hinh.predict_proba(Xb[i:i + 1])
        tre.append(time.perf_counter() - t0)
    p99_tuan.append(1e3 * float(np.percentile(tre, 99)))
    print(f"tuần {so}: xoay {goc:4.0f} độ · độ chính xác {do_cx_tuan[-1]:5.1f} % · p99 {p99_tuan[-1]:.3f} ms")
do_tren_may("độ trễ phân vị 99, tuần một", p99_tuan[0], "ms")
do_tren_may("độ trễ phân vị 99, tuần tám", p99_tuan[-1], "ms")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(cac_tuan, p99_tuan, "o-", color=MAU[0], lw=2)
ax1.set_ylim(0, 2 * max(p99_tuan))
ax1.set_xlabel("tuần")
ax1.set_ylabel("độ trễ phân vị 99 (ms)")
ax1.set_title("Hạ tầng: độ trễ (đo trên máy này)")
ax2.plot(cac_tuan, do_cx_tuan, "o-", color=MAU[1], lw=2)
ax2.set_ylim(0, 100)
ax2.set_xlabel("tuần")
ax2.set_ylabel("độ chính xác (%)")
ax2.set_title("Mô hình: độ chính xác khi nhãn đã về")
for ax in (ax1, ax2):
    ax.axvspan(2.5, 8.5, color=MAU[3], alpha=0.12, lw=0)
    ax.set_xlim(0.5, 8.5)
ax2.text(2.7, 8, "ảnh lệch dần (giả định)", fontsize=8, color="0.3")
plt.tight_layout()
plt.show()

Độ trễ dao động quanh một mức, vì mô hình làm đúng chừng ấy phép nhân cho mỗi yêu cầu, ảnh xoay bao
nhiêu độ cũng vậy. Mọi bảng điều khiển hạ tầng vẫn xanh. Độ chính xác thì đi xuống, nhưng hình bên phải
là thứ chỉ vẽ được **sau khi** nhãn về. Phần còn lại của sổ tay hỏi hai câu: có tín hiệu nào không cần
nhãn mà thấy được sớm hơn, và cần bao nhiêu nhãn để chắc chắn.

## 2. PSI bằng tay: bảng 20

Phương trình 12 so tỉ lệ thực tế (lúc phục vụ) với tỉ lệ kỳ vọng (lúc huấn luyện) trên $n$ nhóm cố định:

$$\text{PSI} = \sum_{i=1}^{n} (\text{thực tế}_i - \text{kỳ vọng}_i) \times \ln\frac{\text{thực tế}_i}{\text{kỳ vọng}_i}$$

Bảng 20 tách PSI của đặc trưng tuổi người dùng qua sáu nhóm tuổi. Phương trình 13 định nghĩa phân kỳ KL
(KL divergence), một phép so khác cho phân phối rời rạc, và nó **không đối xứng**:

$$\mathcal{D}_{\text{KL}}(p \,\|\, q) = \sum_x p(x) \log\frac{p(x)}{q(x)}$$

**Dự đoán:** sách nói vài nhóm dịch tới 3 điểm phần trăm mà PSI tổng vẫn chỉ 0.029, dưới xa ngưỡng cảnh
báo 0.1. Muốn PSI chạm 0.1 thì mỗi nhóm phải dịch gấp mấy lần như thế?

In [ ]:
NHOM_TUOI = ["18–25", "26–35", "36–45", "46–55", "56–65", "66+"]
KY_VONG = np.array([sach(15, nguon="15/12"), sach(25, nguon="25/22"), sach(20, nguon="20/18"),
                    sach(18, nguon="18/20"), sach(12, nguon="12/15"), sach(10, nguon="10/13")]) / 100
THUC_TE = np.array([sach(12, nguon="15/12"), sach(22, nguon="25/22"), sach(18, nguon="20/18"),
                    sach(20, nguon="18/20"), sach(15, nguon="12/15"), sach(13, nguon="10/13 phần trăm")]) / 100


def psi(thuc_te, ky_vong):
    """Phương trình 12. Hai vector tỉ lệ dương, cùng nhóm, mỗi vector cộng lại bằng 1."""
    return float(np.sum((thuc_te - ky_vong) * np.log(thuc_te / ky_vong)))


def kl(p, q):
    """Phương trình 13, logarit tự nhiên."""
    return float(np.sum(p * np.log(p / q)))


print("nhóm     huấn luyện  phục vụ   hiệu    ln(tỉ số)  đóng góp")
for ten, e, a in zip(NHOM_TUOI, KY_VONG, THUC_TE):
    print(f"{ten:<8} {e:>8.0%} {a:>9.0%} {a - e:>+7.2f} {np.log(a / e):>+10.3f} {(a - e) * np.log(a / e):>9.4f}")
theo_sach("PSI tổng của sáu nhóm tuổi", psi(THUC_TE, KY_VONG), sach=0.029, nguon="tổng 0.029")
theo_sach("dịch chuyển lớn nhất của một nhóm (điểm phần trăm)", 100 * np.abs(THUC_TE - KY_VONG).max(),
          sach=3, nguon="3 điểm phần trăm")

# PSI chính là tổng của hai chiều KL: một đẳng thức đại số, không phải xấp xỉ.
kl_di, kl_ve = kl(THUC_TE, KY_VONG), kl(KY_VONG, THUC_TE)
assert abs(psi(THUC_TE, KY_VONG) - (kl_di + kl_ve)) < 1e-12
print(f"\nKL(phục vụ ‖ huấn luyện) = {kl_di:.5f}, KL(huấn luyện ‖ phục vụ) = {kl_ve:.5f}, tổng = PSI")

PSI là phân kỳ KL tính theo cả hai chiều rồi cộng lại, nên nó đối xứng còn KL thì không. Khi dịch chuyển
nhỏ, mỗi đóng góp xấp xỉ $(\text{thực tế}_i - \text{kỳ vọng}_i)^2 / \text{kỳ vọng}_i$, nên PSI tăng theo
**bình phương** độ dịch chuyển: gấp đôi độ dịch thì PSI gấp khoảng bốn. Ô dưới nhân độ dịch của bảng 20
lên $f$ lần và tìm $f$ để chạm hai ngưỡng.

In [ ]:
hieu = THUC_TE - KY_VONG


def psi_nhan(f):
    return psi(KY_VONG + f * hieu, KY_VONG)


cac_f = np.linspace(0, 4.5, 200)
f_canh_bao = optimize.brentq(lambda f: psi_nhan(f) - PSI_CANH_BAO, 0.1, 4.5)
f_bao_dong = optimize.brentq(lambda f: psi_nhan(f) - PSI_BAO_DONG, 0.1, 4.5)
for ten, f in (("cảnh báo 0.1", f_canh_bao), ("báo động 0.25", f_bao_dong)):
    print(f"chạm {ten}: độ dịch gấp {f:.2f} lần bảng 20, tức nhóm dịch nhiều nhất đi {3 * f:.1f} điểm phần trăm")

# Chính sách cho nhóm rỗng: một nhóm "dưới 18" lúc huấn luyện không có ai, lúc phục vụ có 2 phần trăm (giả định).
ky_vong_moi = np.concatenate([[0.0], KY_VONG])
thuc_te_moi = np.concatenate([[0.02], 0.98 * THUC_TE])
print("\nmột nhóm mới xuất hiện, PSI tuỳ cách làm trơn nhóm rỗng:")
for eps in (1e-8, 1e-6, 1e-4, 1e-2):
    e = np.maximum(ky_vong_moi, eps)
    print(f"  thay 0 bằng {eps:g}: PSI = {psi(thuc_te_moi, e / e.sum()):.3f}")

Cùng một dữ liệu mà PSI đi từ dưới 0.1 tới trên 0.25 chỉ vì cách xử lý một nhóm rỗng. Đó là lý do sách
đòi một chính sách cho nhóm rỗng được ghi lại, và ranh giới nhóm được giữ nguyên giữa các kỳ so sánh.

**Trôi chậm.** Sách kể một đặc trưng tuổi qua được mọi kiểm tra giản đồ trong khi trung bình của nó trôi
từ 32 lên 45 trong ba tháng. Ô dưới dựng một phân phối tuổi như thế (giả định: tuổi bằng 18 cộng một biến
Gamma, trung bình tăng đều mỗi tuần trong 13 tuần), chia nhóm theo thập phân vị của tuần đầu, rồi tính PSI
và thống kê KS của từng tuần so với tuần đầu. Các phân phối ở đây là chính xác, không lấy mẫu.

In [ ]:
TUOI_DAU = sach(32, nguon="32 → 45")
TUOI_CUOI = sach(45, nguon="32 → 45")
SO_TUAN = 13                                  # ba tháng, giả định 13 tuần
K_GAMMA = 2.0                                 # giả định về hình dạng phân phối
tb_tuan = np.linspace(TUOI_DAU, TUOI_CUOI, SO_TUAN + 1)
phan_phoi = [stats.gamma(K_GAMMA, loc=18, scale=(m - 18) / K_GAMMA) for m in tb_tuan]
canh = phan_phoi[0].ppf(np.linspace(0, 1, 11)[1:-1])          # thập phân vị của tuần đầu
luoi = np.linspace(18, 200, 4000)


def ti_le_nhom(pp):
    c = np.concatenate([[0.0], pp.cdf(canh), [1.0]])
    return np.diff(c)


psi_tuan = np.array([psi(ti_le_nhom(pp), ti_le_nhom(phan_phoi[0])) for pp in phan_phoi])
ks_tuan = np.array([np.abs(pp.cdf(luoi) - phan_phoi[0].cdf(luoi)).max() for pp in phan_phoi])
nguong_sach = [("PSI 0.1, cảnh báo (bảng 19)", PSI_CANH_BAO),
               ("PSI 0.2, điều tra ở ví dụ gian lận", sach(0.2, nguon="PSI 0.2")),
               ("PSI 0.25, báo động (bảng 19)", PSI_BAO_DONG),
               ("PSI 0.3, mức P2 (bảng 22)", sach(0.3, nguon="PSI trên 0.3"))]
for ten, nguong in nguong_sach:
    vuot = np.nonzero(psi_tuan > nguong)[0]
    print(f"{ten:<36} vượt từ tuần {vuot[0] if len(vuot) else '—'}")
vuot_ks = np.nonzero(ks_tuan > KS_NGUONG)[0]
print(f"{'KS 0.1 (bảng 19)':<36} vượt từ tuần {vuot_ks[0] if len(vuot_ks) else '—'}")
print("kiểm tra hằng tháng (tuần 4, 8, 13): PSI =", ", ".join(f"{psi_tuan[w]:.3f}" for w in (4, 8, 13)))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(3 * cac_f, [psi_nhan(f) for f in cac_f], color=MAU[0], lw=2.2)
for nguong, nhan in ((PSI_CANH_BAO, "cảnh báo 0.1"), (PSI_BAO_DONG, "báo động 0.25")):
    ax1.axhline(nguong, color=XAM, lw=0.9, ls="--")
    ax1.text(0.1, nguong + 0.012, nhan, fontsize=8, color="0.3")
ax1.plot([3], [psi(THUC_TE, KY_VONG)], "o", color=MAU[1], ms=8, zorder=5)
ax1.annotate("bảng 20: 3 điểm, PSI 0.029", (3, psi(THUC_TE, KY_VONG)), xytext=(18, -4), textcoords="offset points",
             fontsize=8, va="center")
ax1.set_xlabel("dịch chuyển của nhóm dịch nhiều nhất (điểm phần trăm)")
ax1.set_ylabel("PSI")
ax1.set_title("Bảng 20, nhân độ dịch lên: PSI tăng theo bình phương")
ax2.plot(np.arange(SO_TUAN + 1), psi_tuan, "o-", color=MAU[0], lw=2, ms=4, label="PSI, 10 nhóm thập phân vị")
ax2.plot(np.arange(SO_TUAN + 1), ks_tuan, "s--", color=MAU[1], lw=1.6, ms=4, label="thống kê KS")
for nguong, nhan in ((PSI_CANH_BAO, "0.1"), (PSI_BAO_DONG, "0.25")):
    ax2.axhline(nguong, color=XAM, lw=0.9, ls="--")
    ax2.text(SO_TUAN, nguong + 0.012, nhan, fontsize=8, color="0.3", ha="right")
ax2.set_xlabel("tuần (trung bình tuổi đi từ 32 lên 45, giả định tăng đều)")
ax2.set_ylabel("so với tuần đầu")
ax2.set_title("Trôi chậm: tuần nào thì vượt ngưỡng")
ax2.legend(loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Với một đặc trưng trôi đều, ngưỡng nào cũng sẽ bị vượt, câu hỏi chỉ là khi nào, và các ngưỡng sách đưa
ra (0.1, 0.2, 0.25, 0.3) rơi vào những tuần khác nhau. Một lịch kiểm tra hằng tháng thấy muộn thêm tối đa
một tháng so với kiểm tra hằng tuần, đúng điều sách gọi là trôi chậm lách qua những ngưỡng thô.

## 3. Bốn kiểu dịch chuyển, bốn bộ theo dõi không cần nhãn

Sách phân biệt hai chiều trôi. Trôi dữ liệu (data drift) là khi phân phối đầu vào $p(x)$ đổi; dịch chuyển
hiệp biến (covariate shift) là trường hợp riêng khi $p(x)$ đổi mà $p(y \mid x)$ giữ nguyên. Trôi khái
niệm (concept drift) là khi chính quan hệ $p(y \mid x)$ đổi. Thêm một kiểu nữa là dịch chuyển nhãn
(label shift): tỉ lệ các lớp đổi. Ô dưới tạo bốn kiểu, mỗi kiểu với mức độ tăng dần:

* **nhiễu**: cộng nhiễu Gauss vào điểm ảnh (dịch chuyển hiệp biến nhẹ);
* **xoay**: xoay ảnh tới 30 độ (dịch chuyển hiệp biến mạnh);
* **tỉ lệ lớp**: chữ số 0 tới 4 xuất hiện nhiều gấp tới 10 lần (dịch chuyển nhãn);
* **khái niệm**: thế giới đổi cách gọi tên, một phần ảnh chữ số 1 giờ mang nhãn 7, 7 thành 1, 3 thành 8
  và 8 thành 3; ảnh không đổi chút nào.

Bốn bộ theo dõi, không bộ nào cần nhãn. Hai bộ nhìn **đầu vào**: KS và PSI trên 5 thành phần chính (PCA)
của ảnh, lấy giá trị lớn nhất trong 5 thành phần. Hai bộ nhìn **đầu ra**: PSI trên phân phối lớp được
dự đoán, một đặc trưng phân loại đúng như bảng 19 dành cho PSI, và KS trên độ tự tin của mô hình (xác suất
lớn nhất nó gán cho một lớp), một đại lượng liên tục đúng như bảng 19 dành cho KS. Mỗi điểm là trung bình
của 10 cửa sổ, mỗi cửa sổ 500 yêu cầu.

**Dự đoán:** bộ nào thấy trôi khái niệm? Có kiểu dịch chuyển nào làm bộ theo dõi báo động mà độ chính xác
không giảm không?

In [ ]:
pca = PCA(5, random_state=SEED).fit(X[hl])
Z_hl = pca.transform(X[hl])
canh_pc = [np.quantile(Z_hl[:, j], np.linspace(0, 1, 11)[1:-1]) for j in range(5)]
P_hl = mo_hinh.predict_proba(X[hl])
lop_hl = np.bincount(P_hl.argmax(1), minlength=10) / len(hl)
tu_tin_hl = P_hl.max(1)
EPS = 1e-4                                     # chính sách nhóm rỗng: thay 0 bằng EPS rồi chuẩn hoá


def psi_dem(dem, ky_vong):
    a = np.maximum(dem / dem.sum(), EPS)
    return psi(a / a.sum(), ky_vong)


ky_vong_pc = [np.bincount(np.searchsorted(canh_pc[j], Z_hl[:, j]), minlength=10) / len(hl) for j in range(5)]
DOI = np.arange(10)
DOI[[1, 7, 3, 8]] = [7, 1, 8, 3]
KIEU = {"nhiễu": ("độ lệch chuẩn của nhiễu", np.linspace(0, 0.4, 9)),
        "xoay": ("góc xoay (độ)", np.linspace(0, 30, 9)),
        "tỉ lệ lớp": ("trọng số của chữ số 0 tới 4", np.linspace(1, 10, 9)),
        "khái niệm": ("tỉ lệ ảnh 1, 7, 3, 8 đổi nhãn", np.linspace(0, 1, 9))}


def cua_so(kieu, muc, n=500, seed=0):
    """Một cửa sổ n yêu cầu với một kiểu dịch chuyển ở một mức. Trả về ảnh và nhãn (nhãn chỉ để chấm)."""
    r = np.random.default_rng(seed)
    if kieu == "tỉ lệ lớp":
        w = np.where(y[kt] <= 4, muc, 1.0)
        chon = r.choice(kt, n, p=w / w.sum())
    else:
        chon = r.choice(kt, n)
    Xb, yb = X[chon].copy(), y[chon].copy()
    if kieu == "nhiễu":
        Xb = np.clip(Xb + r.normal(0, muc, Xb.shape), 0, 1)
    elif kieu == "xoay":
        Xb = xoay(Xb, muc)
    elif kieu == "khái niệm":
        yb = np.where(r.random(n) < muc, DOI[yb], yb)
    return Xb, yb


def theo_doi(Xb, yb):
    P = mo_hinh.predict_proba(Xb)
    Z = pca.transform(Xb)
    return {"độ chính xác": 100 * float((P.argmax(1) == yb).mean()),
            "KS đầu vào": max(stats.ks_2samp(Z_hl[:, j], Z[:, j]).statistic for j in range(5)),
            "PSI đầu vào": max(psi_dem(np.bincount(np.searchsorted(canh_pc[j], Z[:, j]), minlength=10), ky_vong_pc[j])
                               for j in range(5)),
            "PSI lớp dự đoán": psi_dem(np.bincount(P.argmax(1), minlength=10), lop_hl),
            "KS độ tự tin": float(stats.ks_2samp(tu_tin_hl, P.max(1)).statistic)}


print(f"5 thành phần chính giữ {pca.explained_variance_ratio_.sum():.0%} phương sai của 64 điểm ảnh")

In [ ]:
SO_CUA_SO = 10
ket_qua = {}
for kieu, (_, cac_muc) in KIEU.items():
    hang = []
    for muc in cac_muc:
        do = [theo_doi(*cua_so(kieu, muc, seed=k)) for k in range(SO_CUA_SO)]
        hang.append({ten: float(np.mean([d[ten] for d in do])) for ten in do[0]})
    ket_qua[kieu] = hang

# Trôi khái niệm không chạm vào ảnh: bốn bộ theo dõi phải cho đúng giá trị của mức 0, ở mọi mức.
BO_THEO_DOI = ["KS đầu vào", "PSI đầu vào", "PSI lớp dự đoán", "KS độ tự tin"]
for hang in ket_qua["khái niệm"]:
    assert all(hang[ten] == ket_qua["khái niệm"][0][ten] for ten in BO_THEO_DOI)

GIAM = sach(5, nguon="95 phần trăm · 5 điểm phần trăm")    # mức giảm của napkin math 1.4, điểm phần trăm
print(f"mức đầu tiên vượt ngưỡng (PSI 0.1, KS 0.1), và mức đầu tiên độ chính xác giảm từ {GIAM} điểm;")
print("mỗi kiểu đo mức bằng đơn vị của nó: độ lệch chuẩn nhiễu, độ, trọng số, tỉ lệ ảnh đổi nhãn")
print(f"{'':<11}" + "".join(f"{ten:>17}" for ten in BO_THEO_DOI) + f"{'độ chính xác':>15}")
for kieu, (ten_muc, cac_muc) in KIEU.items():
    hang = ket_qua[kieu]
    o = []
    for ten in BO_THEO_DOI:
        nguong = PSI_CANH_BAO if ten.startswith("PSI") else KS_NGUONG
        vuot = [m for m, h in zip(cac_muc, hang) if h[ten] > nguong]
        o.append(f"{vuot[0]:>17.3g}" if vuot else f"{'—':>17}")
    giam = [m for m, h in zip(cac_muc, hang) if h["độ chính xác"] <= hang[0]["độ chính xác"] - GIAM]
    print(f"{kieu:<11}" + "".join(o) + (f"{giam[0]:>15.3g}" if giam else f"{'—':>15}"))

In [ ]:
fig, truc = plt.subplots(2, 4, figsize=(13, 6), sharex="col")
duong = []
for c, (kieu, (ten_muc, cac_muc)) in enumerate(KIEU.items()):
    hang = ket_qua[kieu]
    truc[0, c].plot(cac_muc, [h["độ chính xác"] for h in hang], "o-", color="0.2", lw=2, ms=4)
    truc[0, c].set_ylim(40, 100)
    truc[0, c].set_title(kieu)
    for ten, mau, kieu_net in zip(BO_THEO_DOI, MAU, ("o-", "s--", "^-", "d--")):
        l, = truc[1, c].plot(cac_muc, [max(h[ten], 1e-3) for h in hang], kieu_net, color=mau, lw=1.6, ms=4, label=ten)
        if c == 0:
            duong.append(l)
    truc[1, c].axhline(PSI_CANH_BAO, color=XAM, lw=0.9, ls="--")
    truc[1, c].axhline(PSI_BAO_DONG, color=XAM, lw=0.9, ls=":")
    truc[1, c].set_yscale("log")
    truc[1, c].set_ylim(0.01, 5)
    truc[1, c].set_xlabel(ten_muc)
truc[0, 0].set_ylabel("độ chính xác (%), cần nhãn")
truc[1, 0].set_ylabel("thống kê trôi, không cần nhãn")
truc[1, 3].text(1.0, 0.105, "0.1 ", fontsize=7, color="0.3", va="bottom", ha="right")
truc[1, 3].text(1.0, 0.26, "0.25 ", fontsize=7, color="0.3", va="bottom", ha="right")
fig.suptitle("Bốn kiểu dịch chuyển trên load_digits, trung bình 10 cửa sổ 500 yêu cầu")
fig.legend(handles=duong, loc="lower center", ncol=4, fontsize=8, frameon=False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

Ba điều không phụ thuộc vào máy, vì chúng đến từ chính cách dựng thí nghiệm:

* **Trôi khái niệm vô hình với cả bốn bộ.** Ảnh không đổi, nên mọi thống kê trên ảnh và trên dự đoán đều
  y như ở mức 0 (ô trên đã `assert` điều đó), trong khi độ chính xác giảm. Đúng như sách nói: trôi làm đổi
  chính biên quyết định chỉ phát hiện được khi có nhãn chuẩn.
* **Dịch chuyển nhãn đẩy PSI của lớp dự đoán lên trong khi độ chính xác gần như đứng yên**, vì
  $p(x \mid y)$ không đổi: mô hình vẫn nhận đúng từng chữ số, chỉ là chữ số 0 tới 4 nhiều hơn. Một hệ
  thống tự động huấn luyện lại mỗi khi PSI vượt ngưỡng sẽ huấn luyện lại vô ích ở đây. Đây là lý do sách
  nói vượt ngưỡng PSI nên khởi động điều tra, không phải tự động huấn luyện lại.
* **Mỗi bộ theo dõi nhạy với một kiểu khác nhau.** Bảng trên cho biết, với từng kiểu, bộ nào vượt ngưỡng
  trước mức giảm 5 điểm của độ chính xác. Ngộ nhận cuối chương cho rằng độ chính xác là tín hiệu đầu tiên
  cần theo dõi; với những kiểu mà một bộ không cần nhãn vượt ngưỡng sớm hơn, bộ ấy mua thêm thời gian,
  miễn là nó báo tới đúng người.

## 4. Ngưỡng phụ thuộc vào cỡ cửa sổ

Chú thích của bảng 19 nói ngưỡng KS phải hiệu chuẩn (calibration) theo cỡ mẫu. Lý do: ngay cả khi không có trôi nào,
PSI và KS của một cửa sổ cũng không bằng 0, vì cửa sổ chỉ là một mẫu. Ô dưới mô phỏng 2,000 cửa sổ không
trôi cho mỗi cỡ $n$ và đếm tỉ lệ báo nhầm. Với PSI trên 10 lớp, $n \cdot \text{PSI}$ xấp xỉ phân phối
khi bình phương 9 bậc tự do, nên trung bình của PSI khi không trôi là khoảng $9/n$. Thống kê KS khi không
trôi không phụ thuộc vào phân phối của đặc trưng, nên mô phỏng bằng phân phối đều là đủ.

**Dự đoán:** với cửa sổ 100 yêu cầu, ngưỡng PSI 0.1 báo nhầm bao nhiêu phần trăm số cửa sổ?

In [ ]:
cac_n = np.array([100, 200, 300, 500, 1000, 3000])
m_hl = len(hl)
r = np.random.default_rng(SEED)
mo_phong, ly_thuyet = {}, {}
for n in cac_n:
    psi0 = np.array([psi_dem(r.multinomial(n, lop_hl).astype(float), lop_hl) for _ in range(2000)])
    ks0 = np.array([stats.ks_2samp(r.random(m_hl), r.random(n)).statistic for _ in range(2000)])
    mo_phong[n] = ((psi0 > PSI_CANH_BAO).mean(), (psi0 > PSI_BAO_DONG).mean(), (ks0 > KS_NGUONG).mean(), psi0.mean())
    n_hd = n * m_hl / (n + m_hl)               # cỡ mẫu hiệu dụng của KS hai mẫu
    ly_thuyet[n] = (stats.chi2.sf(n * PSI_CANH_BAO, 9), stats.chi2.sf(n * PSI_BAO_DONG, 9),
                    stats.kstwobign.sf(KS_NGUONG * np.sqrt(n_hd)))
print("   n   PSI>0.1 (xấp xỉ)   PSI>0.25 (xấp xỉ)   KS>0.1 (xấp xỉ)   PSI trung bình vs 9/n")
for n in cac_n:
    a, b, c, tb = mo_phong[n]
    la, lb, lc = ly_thuyet[n]
    print(f"{n:>5}   {a:6.1%} ({la:5.1%})    {b:6.1%} ({lb:5.1%})    {c:6.1%} ({lc:5.1%})    {tb:.4f} vs {9 / n:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.9))
for i, (ten, mau, dau) in enumerate((("PSI > 0.1", MAU[0], "o"), ("PSI > 0.25", MAU[1], "s"), ("KS > 0.1", MAU[2], "^"))):
    ax.plot(cac_n, [100 * ly_thuyet[n][i] for n in cac_n], "-", color=mau, lw=1.6)
    ax.plot(cac_n, [100 * mo_phong[n][i] for n in cac_n], dau, color=mau, ms=7, label=f"{ten}: mô phỏng (điểm), xấp xỉ (đường)")
ax.set_xscale("log")
ax.set_xticks(cac_n, [f"{n:,}" for n in cac_n])
ax.set_xlabel("số yêu cầu trong một cửa sổ (thang log)")
ax.set_ylabel("cửa sổ báo nhầm (%)")
ax.set_title("Không có trôi nào, ngưỡng của bảng 19 vẫn báo khi cửa sổ nhỏ")
ax.legend(loc="upper right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Cửa sổ nhỏ báo nhầm thường xuyên; cửa sổ lớn gần như không bao giờ. Vì thế ngưỡng phải gắn với cỡ cửa sổ:
một hệ thống ít lưu lượng hoặc phải gộp cửa sổ dài hơn, hoặc phải nâng ngưỡng, và cả hai đều làm nó thấy
trôi muộn hơn. Chiều ngược lại cũng là một cái bẫy: PSI và thống kê KS đo **độ lớn** của dịch chuyển, còn
giá trị p của kiểm định KS thì đo **bằng chứng**, nên với đủ nhiều yêu cầu, một dịch chuyển nhỏ tới mức vô
hại cũng có ý nghĩa thống kê. Ô dưới giữ nguyên một dịch chuyển rất nhỏ và chỉ tăng $n$: thống kê KS tiến về
độ lớn thật của dịch chuyển, dưới xa ngưỡng 0.1, còn giá trị p tiến về 0.

In [ ]:
DICH = 0.02                                   # dịch trung bình 0.02 độ lệch chuẩn, giả định là vô hại
D_that = stats.norm.cdf(DICH / 2) - stats.norm.cdf(-DICH / 2)   # thống kê KS của chính hai phân phối
print(f"thống kê KS giữa hai phân phối chuẩn lệch nhau {DICH} độ lệch chuẩn: {D_that:.4f}")
r_p = np.random.default_rng(SEED)
for n in (1_000, 10_000, 100_000, 1_000_000):
    kq = stats.ks_2samp(r_p.normal(0, 1, n), r_p.normal(DICH, 1, n))
    print(f"n = {n:>9,} mỗi phía: thống kê KS = {kq.statistic:.4f}, giá trị p = {kq.pvalue:.2g}")

## 5. Napkin math 1.4: 1,000 nhãn, và vì sao lại là 1,000

Một mô hình đúng 95 phần trăm có thể giảm 5 điểm phần trăm. Chính sách theo dõi đặt ngân sách bằng chứng
là 1,000 mẫu có nhãn. Ở 1 nhãn mỗi giây, thu đủ mất khoảng 16.7 phút; ở 100 nhãn mỗi ngày, mất 10 ngày.
Sách nói rõ con số thật phụ thuộc vào kiểm định, lực kiểm định, nhiễu nhãn và sự phụ thuộc giữa các nhãn.
Ô dưới tính con số thật cho một lựa chọn cụ thể: kiểm định nhị thức một phía, báo nhầm 5 phần trăm, lực
kiểm định 80 phần trăm (giả định của sổ tay).

**Dự đoán:** để thấy mức giảm 5 điểm, cần nhiều hơn hay ít hơn 1,000 nhãn? Còn mức giảm 1 điểm?

In [ ]:
P0 = sach(95, nguon="95 phần trăm · 5 điểm phần trăm") / 100
NGAN_SACH = sach(1_000, nguon="1,000 nhãn")
NHANH = sach(1, nguon="1 mỗi giây")            # nhãn mỗi giây
CHAM = sach(100, nguon="100 mỗi ngày")         # nhãn mỗi ngày
theo_sach("thu 1,000 nhãn ở 1 nhãn mỗi giây (phút)", NGAN_SACH / NHANH / 60, sach=16.7, nguon="1,000 giây ≈ 16.7 phút",
          tol=0.05)
theo_sach("thu 1,000 nhãn ở 100 nhãn mỗi ngày (ngày)", NGAN_SACH / CHAM, sach=10, nguon="10 ngày")

ALPHA, LUC = 0.05, 0.80                        # giả định của sổ tay


def n_xap_xi(p0, p1, alpha=ALPHA, luc=LUC):
    """Cỡ mẫu cho kiểm định một phía H0: p = p0 so với p = p1 < p0, xấp xỉ chuẩn."""
    za, zb = stats.norm.ppf(1 - alpha), stats.norm.ppf(luc)
    return ((za * np.sqrt(p0 * (1 - p0)) + zb * np.sqrt(p1 * (1 - p1))) / (p0 - p1)) ** 2


def n_chinh_xac(p0, p1, alpha=ALPHA, luc=LUC, n_max=20_000):
    """Cỡ mẫu nhỏ nhất mà kiểm định nhị thức chính xác đạt lực kiểm định mong muốn."""
    n = np.arange(10, n_max)
    c = stats.binom.ppf(alpha, n, p0) - 1                     # báo khi số câu đúng <= c
    c = np.where(stats.binom.cdf(c + 1, n, p0) <= alpha, c + 1, c)
    return int(n[np.argmax(stats.binom.cdf(c, n, p1) >= luc)])


print(f"\nkiểm định nhị thức một phía, báo nhầm {ALPHA:.0%}, lực kiểm định {LUC:.0%} (giả định):")
print("mức giảm           nhãn cần (chính xác / xấp xỉ)   ở 1 nhãn mỗi giây   ở 100 nhãn mỗi ngày")
truong_hop = [("95 → 90 (sách)", P0, P0 - GIAM / 100), ("95 → 93", P0, P0 - 0.02), ("95 → 94", P0, P0 - 0.01),
              ("94 → 88 (tuần hai)", 0.94, 0.88)]
for ten, p0, p1 in truong_hop:
    nc = n_chinh_xac(p0, p1)
    print(f"{ten:<20} {nc:>8,} / {n_xap_xi(p0, p1):>8,.0f}         {nc / NHANH / 60:>8.1f} phút      {nc / CHAM:>8.1f} ngày")

# Kiểm bằng mô phỏng: 20,000 cửa sổ 1,000 nhãn, cùng ngưỡng báo của kiểm định chính xác.
c_1000 = stats.binom.ppf(ALPHA, NGAN_SACH, P0) - 1
c_1000 += stats.binom.cdf(c_1000 + 1, NGAN_SACH, P0) <= ALPHA
for p_that, ten in ((P0, "không giảm, báo nhầm"), (P0 - 0.02, "giảm 2 điểm, phát hiện")):
    sim = (r.binomial(NGAN_SACH, p_that, 20_000) <= c_1000).mean()
    print(f"1,000 nhãn, {ten}: mô phỏng {sim:.1%}, chính xác {stats.binom.cdf(c_1000, NGAN_SACH, p_that):.1%}")

In [ ]:
cac_n_luc = np.unique(np.logspace(1, 4, 300).astype(int))
za = stats.norm.ppf(1 - ALPHA)
fig, ax = plt.subplots(figsize=(8, 3.9))
for (ten, p0, p1), mau in zip(truong_hop[:3], MAU):
    luc = stats.norm.cdf(((p0 - p1) * np.sqrt(cac_n_luc) - za * np.sqrt(p0 * (1 - p0))) / np.sqrt(p1 * (1 - p1)))
    ax.plot(cac_n_luc, 100 * luc, color=mau, lw=2)
    i = np.argmax(luc >= 0.5)
    ax.text(cac_n_luc[i] * 1.12, 50, ten.replace(" (sách)", ""), fontsize=8, color="0.2", ha="left")
ax.axhline(100 * LUC, color=XAM, lw=0.9, ls="--")
ax.text(12, 100 * LUC + 2, "lực kiểm định 80 % (giả định)", fontsize=8, color="0.3")
ax.axvline(NGAN_SACH, color="0.2", lw=1.0, ls=":")
ax.text(NGAN_SACH * 1.08, 8, "ngân sách 1,000 nhãn\ncủa sách", fontsize=8, color="0.2")
ax.set_xscale("log")
ax.set_ylim(0, 102)
ax.set_xlabel("số nhãn trong cửa sổ (thang log)")
ax.set_ylabel("xác suất phát hiện (%)")
ax.set_title("Lực kiểm định của kiểm định một phía, từ độ chính xác 95 % (xấp xỉ chuẩn)")
plt.tight_layout()
plt.show()

Với giả định này, 1,000 nhãn dư sức cho mức giảm 5 điểm, vừa đủ cho 2 điểm, và không đủ cho 1 điểm. Vế
"nhãn về chậm" của sách vẫn đúng nguyên: con số cần thiết chia cho tốc độ nhãn về mới ra thời gian chờ.
Kịch bản tuần hai cho một bài học khác: mức giảm 6 điểm cần ít nhãn tới mức, ngay cả ở 100 nhãn mỗi ngày,
vài ngày là đủ, nên sáu tuần chậm trễ ở đó không phải do thống kê, mà do không có hệ thống nào theo dõi.

Hai thứ làm con số thật lớn hơn. **Nhiễu nhãn**: nếu một phần nhãn sai, độ chính xác đo được bị kéo về
phía ngẫu nhiên và mức giảm nhìn thấy nhỏ đi. **Sự phụ thuộc**: nhãn đến theo cụm, ví dụ nhiều nhãn của
cùng một người dùng, mang ít thông tin hơn số lượng của chúng. Với cụm $m$ nhãn và hệ số tương quan nội cụm
$\rho$, phương sai nhân lên $1 + (m-1)\rho$, gọi là hệ số thiết kế. Ô dưới tính cả hai, rồi kiểm hệ số
thiết kế bằng mô phỏng.

In [ ]:
NHIEU_NHAN = 0.10                              # giả định: 10 phần trăm nhãn bị gán nhầm sang lớp khác


def do_duoc(a, eta=NHIEU_NHAN, so_lop=10):
    """Độ chính xác đo được khi một phần eta nhãn sai, rơi đều vào các lớp khác."""
    return a * (1 - eta) + (1 - a) * eta / (so_lop - 1)


p0_n, p1_n = do_duoc(P0), do_duoc(P0 - GIAM / 100)
print(f"nhiễu nhãn {NHIEU_NHAN:.0%}: đo được {p0_n:.4f} → {p1_n:.4f}, mức giảm nhìn thấy {100 * (p0_n - p1_n):.2f} điểm;"
      f" cần {n_chinh_xac(p0_n, p1_n):,} nhãn thay vì {n_chinh_xac(P0, P0 - GIAM / 100):,}")

M_CUM, RHO = 10, 0.10                          # giả định: 10 nhãn mỗi người dùng, tương quan nội cụm 0.1
he_so = 1 + (M_CUM - 1) * RHO
tong_ab = 1 / RHO - 1                          # Beta(a, b) có tương quan nội cụm 1 / (a + b + 1)
so_cum = NGAN_SACH // M_CUM
p_nguoi = r.beta(P0 * tong_ab, (1 - P0) * tong_ab, size=(20_000, so_cum))
dung_cum = r.binomial(M_CUM, p_nguoi).sum(axis=1) / NGAN_SACH
sd_doc_lap = np.sqrt(P0 * (1 - P0) / NGAN_SACH)
print(f"{so_cum} người dùng × {M_CUM} nhãn: độ lệch chuẩn của độ chính xác đo được {dung_cum.std():.4f},"
      f" nếu độc lập {sd_doc_lap:.4f}")
print(f"phương sai gấp {(dung_cum.std() / sd_doc_lap) ** 2:.2f} lần (mô phỏng), hệ số thiết kế {he_so:.2f}:"
      f" 1,000 nhãn theo cụm chỉ đáng {NGAN_SACH / he_so:,.0f} nhãn độc lập")

## 6. Lát cắt: bảng 23, và một lát cắt 5 phần trăm

Bảng 23: người dùng máy tính bảng chỉ chiếm 5 phần trăm lưu lượng nhưng độ chính xác chỉ còn 62 phần trăm,
và độ chính xác tổng thể 91 phần trăm che mất điều đó.

In [ ]:
LAT = ["máy bàn", "iOS", "Android", "máy tính bảng"]
LUU_LUONG = np.array([sach(45, nguon="45/94"), sach(30, nguon="30/92"), sach(20, nguon="20/88"), sach(5, nguon="5/62")]) / 100
DO_CX_LAT = np.array([sach(94, nguon="45/94"), sach(92, nguon="30/92"), sach(88, nguon="20/88"), sach(62, nguon="5/62")])
assert abs(LUU_LUONG.sum() - 1) < 1e-12
tong = float(LUU_LUONG @ DO_CX_LAT)
theo_sach("độ chính xác tổng thể, trung bình theo lưu lượng (phần trăm)", tong, sach=91, nguon="100/91 phần trăm")
print(f"   tính được {tong:.1f}; bảng làm tròn thành 91")
khong = float(LUU_LUONG[:3] @ DO_CX_LAT[:3])
print(f"nếu máy tính bảng sai hết: tổng thể vẫn {khong:.1f} phần trăm")

nhan_lat = NGAN_SACH * LUU_LUONG[3]
ban_kinh = 1.96 * np.sqrt(0.62 * 0.38 / nhan_lat)
print(f"\ntrong 1,000 nhãn, máy tính bảng có {nhan_lat:.0f}: khoảng tin cậy 95 % của 62 % là ±{100 * ban_kinh:.1f} điểm")
can_lat = n_chinh_xac(0.94, 0.88)
print(f"để thấy một lát cắt rơi từ 94 xuống 88 cần {can_lat:,} nhãn của chính lát ấy,"
      f" tức {can_lat / LUU_LUONG[3]:,.0f} nhãn tổng nếu lát cắt chiếm 5 %")

Giờ giấu một lát cắt như thế vào lưu lượng thật. Trong 4,000 yêu cầu, 5 phần trăm đến từ "máy tính bảng",
nơi ảnh bị xoay 25 độ (giả định của sổ tay: một khác biệt trong cách thiết bị chụp ảnh). Ô dưới đo độ chính
xác và hai bộ theo dõi đầu ra trên toàn bộ lưu lượng, trên phần còn lại, và trên riêng lát cắt.

In [ ]:
r_lat = np.random.default_rng(SEED)
chon = r_lat.choice(kt, 4000)
la_bang = r_lat.random(len(chon)) < LUU_LUONG[3]
Xs = X[chon].copy()
Xs[la_bang] = xoay(Xs[la_bang], 25.0)
P = mo_hinh.predict_proba(Xs)
for ten, mat_na in (("toàn bộ", np.ones_like(la_bang)), ("không phải máy tính bảng", ~la_bang),
                    ("máy tính bảng", la_bang)):
    dung = 100 * float((P[mat_na].argmax(1) == y[chon][mat_na]).mean())
    psi_l = psi_dem(np.bincount(P[mat_na].argmax(1), minlength=10).astype(float), lop_hl)
    ks_l = stats.ks_2samp(tu_tin_hl, P[mat_na].max(1)).statistic
    print(f"{ten:<26} {mat_na.sum():>5} yêu cầu · PSI lớp dự đoán {psi_l:.3f} · KS độ tự tin {ks_l:.3f}")
    do_tren_may(f"độ chính xác, {ten}", dung, "%")

Trên toàn bộ lưu lượng, cả độ chính xác lẫn hai bộ theo dõi đều bị pha loãng bởi 95 phần trăm lưu lượng
khoẻ mạnh. Tách riêng lát cắt thì vấn đề lộ ra, nhưng lát cắt chỉ có khoảng 200 yêu cầu, đúng vùng mà
phần 4 cho thấy ngưỡng bắt đầu báo nhầm, và đúng vùng mà phần 5 cho thấy nhãn không đủ. Đây là cái giá của
phân tích theo lát cắt: càng chia nhỏ để thấy, mỗi lát càng ít dữ liệu để tin.

## Thử thêm

1. Ở phần 3, thay `pca.transform(Xb)` bằng chính `Xb` (64 điểm ảnh, bỏ PCA) trong `theo_doi`, đổi
   `range(5)` thành `range(64)` và dựng lại `canh_pc`, `ky_vong_pc`, `Z_hl` theo điểm ảnh. Nhiều điểm ảnh ở
   rìa luôn bằng 0 lúc huấn luyện: PSI và KS đầu vào phản ứng với nhiễu ra sao?
2. Ở phần 3, đặt `n=100` trong `cua_so`. Ở mức 0, bộ nào đã vượt ngưỡng? So với bảng của phần 4.
3. Ở phần 5, đặt `RHO = 0.3`. Ngân sách 1,000 nhãn còn đáng bao nhiêu nhãn độc lập, và còn đủ cho mức giảm
   5 điểm không?

## Tóm lại

* Độ trễ không đổi khi dữ liệu dịch chuyển, vì phép tính không đổi; độ chính xác thì chỉ đo được khi nhãn về.
* PSI là phân kỳ KL tính theo hai chiều, tăng theo bình phương độ dịch; bảng 20 cho 0.029, và cách xử lý
  một nhóm rỗng có thể đổi kết luận.
* Không bộ theo dõi nào không cần nhãn thấy được trôi khái niệm; dịch chuyển nhãn đẩy PSI lên mà không làm
  giảm độ chính xác. Vượt ngưỡng là lý do để điều tra, không phải để tự động huấn luyện lại.
* Ngưỡng của bảng 19 báo nhầm thường xuyên ở cửa sổ nhỏ; ngưỡng phải gắn với cỡ cửa sổ.
* Với kiểm định một phía và lực kiểm định 80 phần trăm, 1,000 nhãn dư cho mức giảm 5 điểm nhưng không đủ cho
  1 điểm; nhiễu nhãn và nhãn theo cụm làm con số thật lớn hơn, và lát cắt nhỏ thì cần nhiều nhãn hơn nữa.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Vận hành học máy — khi hệ thống sẵn sàng mà vẫn sai](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch14-ml-ops/), dựng từ chương
[*ML Operations*](https://mlsysbook.ai/vol1/ml_ops/ml_ops.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_14_ml_ops.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.